In [ ]:
#| default_exp meta_learning.utils.tb_logger

In [ ]:
#| export

import datetime
import random
import json
import os
import wandb
import torch
#from torch.utils.tensorboard import SummaryWriter

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:

# class TBLogger:
#     def __init__(self, args, exp_label):
#         self.output_name = exp_label + '_' + str(args.seed) + '_' + datetime.datetime.now().strftime('_%d:%m_%H:%M:%S') + '_' + hex(random.Random().randint(0,1e6)).replace("0x","")
#         try:
#             log_dir = args.results_log_dir
#         except AttributeError:
#             log_dir = args['results_log_dir']

#         if log_dir is None:
#             dir_path = os.path.abspath(os.path.join(os.path.dirname(os.path.realpath(__file__)), os.pardir))
#             dir_path = os.path.join(dir_path, 'logs')
#         else:
#             dir_path = log_dir

#         if not os.path.exists(dir_path):
#             try:
#                 os.mkdir(dir_path)
#             except FileExistsError: # This can still happen in the intervening time
#                 pass
#             except:
#                 dir_path_head, dir_path_tail = os.path.split(dir_path)
#                 if len(dir_path_tail) == 0:
#                     dir_path_head, dir_path_tail = os.path.split(dir_path_head)
#                 os.mkdir(dir_path_head)
#                 os.mkdir(dir_path)

#         try:
#             self.full_output_folder = os.path.join(os.path.join(dir_path, 'logs_{}'.format(args.env_name)),
#                                                    self.output_name)
#         except:
#             self.full_output_folder = os.path.join(os.path.join(dir_path, 'logs_{}'.format(args["env_name"])),
#                                                    self.output_name)

#         self.writer = SummaryWriter(log_dir=self.full_output_folder)

#         print('logging under', self.full_output_folder)

#         if not os.path.exists(self.full_output_folder):
#             os.makedirs(self.full_output_folder)
#         with open(os.path.join(self.full_output_folder, 'config.json'), 'w') as f:
#             try:
#                 config = {k: v for (k, v) in vars(args).items() if k != 'device'}
#             except:
#                 config = args
#             config.update(device=device.type)
#             json.dump(config, f, indent=2)

#     def add(self, name, value, x_pos):
#         self.writer.add_scalar(name, value, x_pos)

In [ ]:

#| export

class WandbLogger:
    def __init__(self, args, exp_label):
        self.output_name = exp_label + '_' + str(args.seed) + '_' + datetime.datetime.now().strftime('_%d:%m_%H:%M:%S') + '_' + hex(random.Random().randint(0,1e6)).replace("0x","")
        
        try:
            log_dir = args.results_log_dir
        except AttributeError:
            log_dir = ".results"

        if log_dir is None:
            dir_path = os.path.abspath(os.path.join(os.path.dirname(os.path.realpath(__file__)), os.pardir))
            dir_path = os.path.join(dir_path, 'logs')
        else:
            dir_path = log_dir

        self.full_output_folder = os.path.join(os.path.join(dir_path, f'logs_{args.env_name}'), self.output_name)
        os.makedirs(self.full_output_folder, exist_ok=True)
        
        # Prepare config
        try:
            config = {k: v for (k, v) in vars(args).items() if k != 'device'}
        except:
            config = args
        config.update(device=device.type)
        
        # Save config
        with open(os.path.join(self.full_output_folder, 'config.json'), 'w') as f:
            json.dump(config, f, indent=2)

        # Start wandb
        wandb.init(project="hyper-meta-rl", name=self.output_name, config=config, dir=self.full_output_folder)
        print('Logging to Weights & Biases:', self.full_output_folder)
        # Tell wandb we have *two* step metrics
        wandb.define_metric("iter")
        wandb.define_metric("frame")
        # Any metric whose name starts with this prefix will use that step
        wandb.define_metric("return_avg_per_iter/*",  step_metric="iter")
        wandb.define_metric("return_std_per_iter/*",  step_metric="iter")
        wandb.define_metric("return_avg_per_frame/*", step_metric="frame")
        wandb.define_metric("return_std_per_frame/*", step_metric="frame")
        wandb.define_metric("Meta-Episode Return",    step_metric="frame")

    
        self._iter_counter = 0
    def add(self, name, value, x_pos):
        # choose which step metric to log
        if "iter" in name:
            wandb.log({"iter": x_pos, name: value})
        elif "frame" in name:
            wandb.log({"frame": x_pos, name: value})
        else:
            # metrics that were already unique per-iteration can
            # just use iter as the step axis
            wandb.log({"iter": self._iter_counter, name: value})
        self._iter_counter += 1

    def close(self):
        wandb.finish()
